In [1]:
from modules import CreationFlavaDataset, creation_dataframe,FlavaExtractor,CreationProcessedDataset,Train,HeadClassifierFlavaModel
from torchvision import transforms
from transformers import FlavaModel,AutoProcessor
from torch.utils.data import DataLoader
import torch
from sklearn.utils.class_weight import compute_class_weight
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from torch.nn.modules.loss import BCEWithLogitsLoss,CrossEntropyLoss
import numpy as np

import sys, os
sys.path.append(os.path.abspath(".."))

from CLIP_model.modules import CreationProcessedDataset as CreationClipDataset


In [2]:
train_df=creation_dataframe("../data/train.jsonl")
val_df=creation_dataframe("../data/dev.jsonl")

In [3]:
train_FLAVA_dataset=CreationFlavaDataset(train_df)
val_FLAVA_dataset=CreationFlavaDataset(val_df)

In [4]:
processor=AutoProcessor.from_pretrained("facebook/flava-full")

In [5]:
flava_model=FlavaModel.from_pretrained("facebook/flava-full")
device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))
batch_size=32

In [6]:
def collate_fn(batch):
    images=[b["images"] for b in batch]
    texts=[b["texts"] for b in batch]
    labels=[b["labels"] for b in batch]
    inputs=processor(images,texts,return_tensors="pt",padding="max_length",truncation=True, max_length=128)
    inputs["labels"]=torch.tensor(labels,dtype=torch.float32)
    return inputs

In [7]:
train_FLAVA_dataloader=DataLoader(train_FLAVA_dataset,collate_fn=collate_fn,batch_size=batch_size,shuffle=False)
val_FLAVA_dataloader=DataLoader(val_FLAVA_dataset,collate_fn=collate_fn,batch_size=batch_size,shuffle=False)

In [ ]:
#flava_extractor=FlavaExtractor(flava_model=flava_model,device=device)

In [ ]:
#train_flava_data,val_flava_data=flava_extractor.get_embeddings(train_FLAVA_dataloader,val_FLAVA_dataloader,"./modules/flava_embeddings")

c:\Users\sh032\anaconda3\Lib\site-packages\transformers\modeling_utils.py:1621: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


Embeddings saved to ./modules/flava_embeddings


In [8]:
train_flava_embeddings=torch.load("./modules/flava_embeddings/train_flava_embeddings.pt")
val_flava_embeddings=torch.load("./modules/flava_embeddings/val_flava_embeddings.pt")

In [11]:
train_dataset=CreationProcessedDataset(train_flava_embeddings)
val_dataset=CreationProcessedDataset(val_flava_embeddings)
train_dataloader=DataLoader(train_dataset,batch_size=batch_size,shuffle=True,drop_last=True)
val_dataloader=DataLoader(val_dataset,batch_size=batch_size,shuffle=True,drop_last=True)

In [12]:
model=HeadClassifierFlavaModel(fc_layer_sizes=[512,384,220])

In [11]:
class_weight=compute_class_weight("balanced",classes=np.unique(train_df["label"]),y=train_df["label"].to_numpy())
class_weight=torch.tensor(class_weight, dtype=torch.float32)
print(class_weight)

tensor([0.7798, 1.3934])


In [12]:
n_epochs=10
n_steps=(train_dataset.__len__()//batch_size)*n_epochs
optimizer=AdamW(model.parameters(),lr=0.01,weight_decay=0.1)
scheduler=get_linear_schedule_with_warmup(optimizer,num_warmup_steps=0.1*n_steps,num_training_steps=n_steps)
loss_fn=CrossEntropyLoss(weight=class_weight)

In [13]:
trainer=Train(processor,model,loss_fn,optimizer,n_epochs,scheduler,device,batch_size, patience=50, min_improvement=0.05)

In [14]:
trainer.run_training(train_dataloader=train_dataloader,val_dataloader=val_dataloader)

2026-02-19 15:17:58.052 | INFO     | modules.Train:run_training:41 - Epoch 0 :
c:\Users\sh032\anaconda3\Lib\site-packages\torch\optim\lr_scheduler.py:192: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(
2026-02-19 15:18:02.945 | INFO     | modules.Train:run_training:117 - Epoch 0: Train Loss = 0.7813393545330019
2026-02-19 15:18:02.945 | INFO     | modules.Train:run_training:118 - Epoch 0: Train Accuracy = 0.6411764705882353
2026-02-19 15:18:02.960 | INFO     | modules.Train:run_training:119 - Epoch 0: Train F1 = 0.0
2026-02-19 15:18:02.960 | INFO     | modules.Train:run_training:121 - Epoch 0: Validation Loss = 2.755837067961693
202

In [13]:
train_clip_data=torch.load("../CLIP_model/modules/clip_embeddings/train_clip_embeddings.pt")
val_clip_data=torch.load("../CLIP_model/modules/clip_embeddings/train_clip_embeddings.pt")

In [14]:
train_clip_dataset=CreationClipDataset(train_clip_data)
val_clip_dataset=CreationClipDataset(val_clip_data)
train_clip_dataloader=DataLoader(train_clip_dataset,batch_size=32,shuffle=True,drop_last=True)
val_clip_dataloader=DataLoader(val_clip_dataset,batch_size=32,shuffle=True,drop_last=True)

In [15]:
model_with_clip=HeadClassifierFlavaModel(fc_layer_sizes=[768,512,384,220],with_clip_image=True,with_clip_text=True)

In [16]:
class_weight=compute_class_weight("balanced",classes=np.unique(train_df["label"]),y=train_df["label"].to_numpy())
class_weight=torch.tensor(class_weight, dtype=torch.float32)
print(class_weight)

tensor([0.7798, 1.3934])


In [17]:
n_epochs=10
n_steps=(train_dataset.__len__()//batch_size)*n_epochs
optimizer=AdamW(model.parameters(),lr=0.01,weight_decay=0.1)
scheduler=get_linear_schedule_with_warmup(optimizer,num_warmup_steps=0.1*n_steps,num_training_steps=n_steps)
loss_fn=CrossEntropyLoss(weight=class_weight)

In [18]:
trainer_with_clip=Train(processor,model_with_clip,loss_fn,optimizer,n_epochs,scheduler,device,batch_size, patience=50, min_improvement=0.05)

In [19]:
trainer_with_clip.run_training(train_dataloader=train_dataloader,val_dataloader=val_dataloader,with_clip=True,train_clip_dataloader=train_clip_dataloader,val_clip_dataloader=val_clip_dataloader)

2026-02-19 15:50:13.112 | INFO     | modules.Train:run_training:41 - Epoch 0 :


c:\Users\sh032\anaconda3\Lib\site-packages\torch\optim\lr_scheduler.py:192: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(
2026-02-19 15:50:32.431 | INFO     | modules.Train:run_training:118 - Epoch 0: Train Loss = 0.7106778183073368
2026-02-19 15:50:32.488 | INFO     | modules.Train:run_training:119 - Epoch 0: Train Accuracy = 0.5448113207547169
2026-02-19 15:50:32.488 | INFO     | modules.Train:run_training:120 - Epoch 0: Train F1 = 0.3568810396534488
2026-02-19 15:50:32.504 | INFO     | modules.Train:run_training:122 - Epoch 0: Validation Loss = 0.7396298964818319
2026-02-19 15:50:32.505 | INFO     | modules.Train:run_training:12